In [ ]:
#import 
import torch
from torch_geometric.loader import DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from itertools import product
import os
from networks import GAT, PPGAT
import os
import json
import pickle
import time
import random
import pandas as pd


In [ ]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
with open('datasets/df_random_vs_st_dataset_revised.pkl', 'rb') as f:
    all_data = pickle.load(f)
targets =all_data['accession'].unique()
targets

array(['P08581', 'P35968', 'Q16790', 'P56817', 'P22303', 'P06276',
       'P00915', 'P34913', 'Q13547', 'P27487'], dtype=object)

In [ ]:
def subset_by_smiles(dataset, smiles_subset):
    smiles_set = set(smiles_subset)
    return [data for data in dataset if data.smiles in smiles_set]


def train_eval_model(
    mod,
    train_dataset,
    val_dataset,
    test_dataset,
    in_channels,
    edge_attr_dim,
    out_channels,
    grid,
    epochs=30,
    device='cuda',
    save_dir="results"
):
    os.makedirs(save_dir, exist_ok=True)  # create folder if it doesn’t exist
    
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    results = [] # timing and loss info

    # Hyperparameter search
    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        #measure total running time 
        total_train_start = time.time()


        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float().unsqueeze(1))
                loss.backward()
                optimizer.step()
            epoch_end = time.time()
        
        total_train_time = time.time() - total_train_start

         

        # Validation loop
        model.eval()
        val_loss = 0

        #inference/validation time
        val_start = time.time()

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                val_loss += criterion(out, batch.y.float().unsqueeze(1)).item() * batch.num_graphs
        val_time = time.time() - val_start

        val_loss /= len(val_dataset)

        throughput = len(train_dataset) / total_train_time


        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #Retrain best model on train + val
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to(device)
    final_model.load_state_dict(best_model_state)

    full_train = train_dataset + val_dataset
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float().unsqueeze(1))
            loss.backward()
            optimizer.step()

    # Save model + config + test set
    model_path = os.path.join(save_dir, "best_model.pt")
    torch.save(final_model.state_dict(), model_path)

    testset_path = os.path.join(save_dir, "test_dataset.pt")
    torch.save(test_dataset, testset_path )


    config_path = os.path.join(save_dir, "best_config.json")
    with open(config_path, "w") as f:
        json.dump({"lr": best_config[0], "batch_size": best_config[1]}, f)



    results.append({
        "lr":lr,
        "batch_size" : batch_size,
        "val_loss": val_loss,
        "train_time_s" : val_time,
        "throuhput_s" : throughput
    })

    results_path = os.path.join(save_dir, "timing_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent="")

    print(f" Model saved to {model_path}")
    print(f" Best config saved to {config_path}")
    print(f" timing results saved to {results_path}")


    return final_model, test_dataset, best_config

In [ ]:
def analogue_series_split(data, seed=42, test_size=0.3, n_splits=10, n_cpds_tolerance=5):
    
    from collections import defaultdict

    scaffolds = defaultdict(list)
    for idx, core in enumerate(data.core):
        scaffolds[core].append(idx)
    
    n_total_test = int(np.floor(test_size * len(data)))
    rng = np.random.RandomState(seed)
    for i in range(n_splits):

        scaffold_list = list(scaffolds.values())
        scaffold_indices = rng.permutation(len(scaffold_list))
        scaffold_sets = [scaffold_list[i] for i in scaffold_indices]

        train_index = []
        test_index = []

        for scaffold_set in scaffold_sets:
            if len(test_index) + len(scaffold_set) <= n_total_test:
                test_index.extend(scaffold_set)
            else:
                train_index.extend(scaffold_set)
        
        # Check for tolerance requirement
        if np.abs(len(test_index) - n_total_test) <= n_cpds_tolerance:
            print(f"Split {i} meets tolerance.")
            print(f"Train proportion: {len(train_index)/len(data):.2f}")
            print(f"Test proportion: {len(test_index)/len(data):.2f}")
            yield train_index, test_index
        else:
            print(f"Warning: Split {i} does not meet tolerance. Test size: {len(test_index)}, Expected: {n_total_test}")
            yield train_index, test_index

In [ ]:
# loop 
trials = 5

for target in targets:
    print(target)

    df_data = pd.read_csv(f"analogue_series_split/df_ccr_results_{target}.csv")

    df_train_all_splits = pd.DataFrame()
    df_test_all_splits = pd.DataFrame()
    
    for trial, (train_index, test_index) in enumerate(
        analogue_series_split(df_data, seed=42, test_size=0.2, n_splits=trials)
    ):
        df_train = df_data.iloc[train_index].copy()
        df_test  = df_data.iloc[test_index].copy()

        df_train["trial"] = trial
        df_test["trial"]  = trial

        df_train_all_splits = pd.concat([df_train_all_splits, df_train])
        df_test_all_splits  = pd.concat([df_test_all_splits, df_test])

        filtered_data = all_data[all_data['accession'] == target]

        # MAIN RUN LOOP 
        for run in range(trials):
            print("RUN:", run)

            train_val_set = df_train_all_splits[df_train_all_splits['trial'] == run] 
            test_set = df_test_all_splits[df_test_all_splits['trial'] == run].copy()

            # get val split
            train_set = pd.DataFrame()
            val_set= pd.DataFrame()

            for trial, (train_index, val_index) in enumerate(
                analogue_series_split(train_val_set, seed=42, test_size=0.1, n_splits=1)
            ):
                df_train = train_val_set.iloc[train_index].copy()
                df_val   = train_val_set.iloc[val_index].copy()

                train_set = pd.concat([train_set, df_train])
                val_set   = pd.concat([val_set, df_val])

            # labels
            test_set["label"] = 1
            train_set["label"] = 1
            val_set["label"] = 1

            # inactive data
            
            inactive_data = filtered_data[filtered_data['label'] == 0].copy()

            inactive_data = inactive_data.rename(columns={
                "chembl_cid": "cid",
                "chembl_tid": "tid",
                "nonstereo_aromatic_smiles": "smiles"
            })

            n_train_active = len(train_set)
            n_test_active  = len(test_set)
            n_val_active   = len(val_set)

            total_needed = n_train_active + n_test_active + n_val_active

            inactive_sample = inactive_data.sample(
                n=total_needed,
                replace=False,
                random_state= 42 + run
            )

            inactive_train = inactive_sample.iloc[:n_train_active]
            inactive_test  = inactive_sample.iloc[n_train_active : n_train_active + n_test_active]
            inactive_val   = inactive_sample.iloc[n_train_active + n_test_active:]

            train_balanced = pd.concat([train_set, inactive_train]).reset_index(drop=True)
            test_balanced  = pd.concat([test_set, inactive_test]).reset_index(drop=True)
            val_balanced   = pd.concat([val_set, inactive_val]).reset_index(drop=True)

            # datasets

            dataset = torch.load(f'datasets/Gdatasets/{target}_dataset.pt', weights_only=False)
            rg_dataset = torch.load(f'datasets/RGdatasets/{target}_RG_dataset.pt', weights_only=False)
            ppgat_dataset = torch.load(f'datasets/PPGATdatasets/{target}_PPGAT_dataset.pt', weights_only=False)


            test_set = subset_by_smiles(dataset, test_balanced.smiles)
            train_set = subset_by_smiles(dataset, train_balanced.smiles)
            val_set = subset_by_smiles(dataset, val_balanced.smiles)

            test_rg = subset_by_smiles(rg_dataset, test_balanced.smiles)
            train_rg = subset_by_smiles(rg_dataset, train_balanced.smiles)
            val_rg = subset_by_smiles(rg_dataset, val_balanced.smiles)

            test_ppgat = subset_by_smiles(ppgat_dataset, test_balanced.smiles)
            train_ppgat = subset_by_smiles(ppgat_dataset, train_balanced.smiles)
            val_ppgat = subset_by_smiles(ppgat_dataset, val_balanced.smiles)
    



            print("GAT")
            in_channels = dataset[0].x.size(-1)
            edge_attr_dim = dataset[0].edge_attr.size(-1)
            out_channels = 1

            #grid search
            learning_rates = [1e-4, 5e-4, 1e-3]
            batch_sizes = [16, 32, 64]
            grid = list(product(learning_rates, batch_sizes))

            os.makedirs('results/models/gat', exist_ok=True)

            gat_model, gat_test, best_config = train_eval_model(
                GAT,
                train_set,
                val_set,
                test_set,
                in_channels,
                edge_attr_dim,
                out_channels,
                grid=grid,
                save_dir = f"results/models/gat/{target}"
            )

            print("GAT-rg")
            in_channels = rg_dataset[0].x.size(-1)
            edge_attr_dim = rg_dataset[0].edge_attr.size(-1)
            out_channels = 1

            #grid search
            learning_rates = [1e-4, 5e-4, 1e-3]
            batch_sizes = [16, 32, 64]
            grid = list(product(learning_rates, batch_sizes))

            os.makedirs('results/models/gat_rg', exist_ok=True)

            gat_rg_model, gat_rg_test, best_config = train_eval_model(
                GAT,
                train_rg,
                val_rg,
                test_rg,
                in_channels,
                edge_attr_dim,
                out_channels,
                grid=grid,
                save_dir = f"results/models/gat_rg/{target}"
            )


            #PPGAT
            print("PPGAT")
            in_channels = ppgat_dataset[0].x.size(-1)
            edge_attr_dim = ppgat_dataset[0].edge_attr.size(-1)
            out_channels = 1

            #grid search
            learning_rates = [1e-4, 5e-4, 1e-3]
            batch_sizes = [16, 32, 64]
            grid = list(product(learning_rates, batch_sizes))
            
            os.makedirs('results/models/ppgat', exist_ok=True)

            rgnn_model, rgnn_test, best_config= train_eval_model(
                PPGAT,
                train_ppgat,
                val_ppgat,
                test_ppgat,
                in_channels,
                edge_attr_dim,
                out_channels,
                grid=grid,
                save_dir = f"results/models/ppgat/{target}"
            )

            print("done")

P08581
GAT
 Model saved to results/models/gat/P08581/best_model.pt
 Best config saved to results/models/gat/P08581/best_config.json
 timing results saved to results/models/gat/P08581/timing_results.json
GAT-rg
 Model saved to results/models/gat_rg/P08581/best_model.pt
 Best config saved to results/models/gat_rg/P08581/best_config.json
 timing results saved to results/models/gat_rg/P08581/timing_results.json
PPGAT
 Model saved to results/models/ppgat/P08581/best_model.pt
 Best config saved to results/models/ppgat/P08581/best_config.json
 timing results saved to results/models/ppgat/P08581/timing_results.json
done
P35968
GAT
 Model saved to results/models/gat/P35968/best_model.pt
 Best config saved to results/models/gat/P35968/best_config.json
 timing results saved to results/models/gat/P35968/timing_results.json
GAT-rg
 Model saved to results/models/gat_rg/P35968/best_model.pt
 Best config saved to results/models/gat_rg/P35968/best_config.json
 timing results saved to results/models/gat